# Project 1 - Retail Analytics

In this notebook, we load a large retail dataset and runs four required algorithms from the `/src` folder:

1. Top-K revenue products
2. Duplicate detection
3. Rolling average revenue
4. Z-score anomaly detection

It also demonstrates secure coding using **logging**, **exception handling**, and **basic validation checks**

In [27]:
import os
import sys
import pandas as pd
import numpy as np
import logging
from scipy import stats  # kept for starter compatibility (even if src uses it)

# Configure logging so we can see info/errors clearly during execution.
logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(message)s")

# Notebook logger (separate from module loggers in /src).
logger = logging.getLogger("notebook")

# TODO: Make sure /src is importable (required project structure: notebooks/ and src/)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

# TODO: Import required functions from /src (do NOT re-implement them here)
from src.revenue_analysis import top_k_revenue_products, rolling_avg_revenue
from src.anomaly_detection import find_duplicates, zscore_anomalies

logger.info("Imported functions from src successfully.")


INFO:Imported functions from src successfully.


## 1) Load the dataset

We load the dataset from the `/data` folder
We use **try/except** so the notebook fails safely with clear error message if the file is missing

In [28]:
# TODO: Ensure retail_orders_large.csv is in the same folder as this notebook.
# In Google Colab, upload the file from the left panel (Files).

csv_path = "../data/retail_orders_large.csv"

try:
    df = pd.read_csv(csv_path)
    logging.info(f"Loaded dataset shape: {df.shape}")
except FileNotFoundError:
    logging.error("CSV not found. Upload retail_orders_large.csv or place it in the notebook folder.")
    raise
except pd.errors.EmptyDataError:
    logging.error("CSV is empty or unreadable.")
    raise
except Exception as e:
    logging.error(f"Unexpected error loading CSV: {e}")
    raise

df.head()


INFO:Loaded dataset shape: (75000, 8)


,order_id,customer_id,product_id,order_date,quantity,unit_price,region,revenue
0,1,25795,229,2022-02-16,5,398.29,West,1991.45
1,2,10860,567,2023-05-25,3,345.75,West,1037.25
2,3,86820,445,2022-02-05,4,419.41,West,1677.64
3,4,64886,933,2024-05-13,5,123.95,North,619.75
4,5,16265,275,2024-02-20,2,286.49,East,572.98


In [19]:
# TODO: Ensure retail_orders_large.csv is in the same folder as this notebook.
# In Google Colab, upload the file from the left panel (Files).

csv_path = "../data/retail_orders_large.csv"

try:
    df = pd.read_csv(csv_path)
    logging.info(f"Loaded dataset shape: {df.shape}")
except FileNotFoundError:
    logging.error("CSV not found. Upload retail_orders_large.csv or place it in the notebook folder.")
    raise
except pd.errors.EmptyDataError:
    logging.error("CSV is empty or unreadable.")
    raise
except Exception as e:
    logging.error(f"Unexpected error loading CSV: {e}")
    raise

df.head()

INFO:Loaded dataset shape: (75000, 8)


,order_id,customer_id,product_id,order_date,quantity,unit_price,region,revenue
0,1,25795,229,2022-02-16,5,398.29,West,1991.45
1,2,10860,567,2023-05-25,3,345.75,West,1037.25
2,3,86820,445,2022-02-05,4,419.41,West,1677.64
3,4,64886,933,2024-05-13,5,123.95,North,619.75
4,5,16265,275,2024-02-20,2,286.49,East,572.98


## 2) Data Validation & Quick EDA

We quickly validate the dataset by checking:
- data types and column info
- missing values
- duplicate rows
- quick numeric summary

In [31]:
# TODO: Print basic info, missing values, duplicates.
# Hint: df.info(), df.isna().sum(), df.duplicated().sum()

# --- YOUR CODE HERE ---

logging.info("=== Basic Dataset Info ===")
df.info()

logging.info("=== Missing Values (per column) ===")
missing = df.isna().sum().sort_values(ascending=False)
display(missing)

logging.info("=== Duplicate Rows Count ===")
dup_count = int(df.duplicated().sum())
print("Duplicate rows:", dup_count)

logging.info("=== Numeric Summary ===")
display(df.describe(include=[np.number]).T)

logging.info("=== Head ===")
display(df.head())


INFO:=== Basic Dataset Info ===
INFO:=== Missing Values (per column) ===


<class 'pandas.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     75000 non-null  int64  
 1   customer_id  75000 non-null  int64  
 2   product_id   75000 non-null  int64  
 3   order_date   75000 non-null  str    
 4   quantity     75000 non-null  int64  
 5   unit_price   75000 non-null  float64
 6   region       75000 non-null  str    
 7   revenue      75000 non-null  float64
dtypes: float64(2), int64(4), str(2)
memory usage: 4.6 MB


order_id       0
customer_id    0
product_id     0
order_date     0
quantity       0
unit_price     0
region         0
revenue        0
dtype: int64

INFO:=== Duplicate Rows Count ===
INFO:=== Numeric Summary ===


Duplicate rows: 0


,count,mean,std,min,25%,50%,75%,max
order_id,75000.0,37500.500000,21650.779432,1.00,18750.7500,37500.500,56250.2500,75000.00
customer_id,75000.0,54940.393773,26141.642334,10001.00,32133.7500,55047.500,77639.5000,99998.00
product_id,75000.0,549.157907,259.277001,100.00,324.0000,550.000,773.0000,998.00
quantity,75000.0,3.004320,1.413132,1.00,2.0000,3.000,4.0000,5.00
unit_price,75000.0,252.769362,143.208260,5.01,128.1500,253.265,376.6800,499.99
revenue,75000.0,759.060178,594.646710,5.01,282.8475,596.880,1124.2125,2499.65


INFO:=== Head ===


,order_id,customer_id,product_id,order_date,quantity,unit_price,region,revenue
0,1,25795,229,2022-02-16,5,398.29,West,1991.45
1,2,10860,567,2023-05-25,3,345.75,West,1037.25
2,3,86820,445,2022-02-05,4,419.41,West,1677.64
3,4,64886,933,2024-05-13,5,123.95,North,619.75
4,5,16265,275,2024-02-20,2,286.49,East,572.98


## 3.1 Algorithm 1 — Top-K Revenue Products

We find the top 10 products by total revenue.  

In [32]:
try:
    top10 = top_k_revenue_products(df, k=10)
    display(top10)
except Exception as e:
    logging.error(f"Failed to compute top-k revenue products: {e}")
    raise


INFO:Computed top-10 revenue products.


,product_id,revenue
0,103,92480.65
1,754,90877.86
2,690,90119.02
3,179,88129.24
4,788,87105.45
5,232,85028.09
6,129,84523.50
7,452,84478.93
8,335,84391.11
9,919,84203.65


## 3.2 Algorithm 2 — Duplicate Detection

In [33]:
try:
    dups = find_duplicates(df)
    print("Duplicate rows (keep all duplicates):", len(dups))
    display(dups.head(10))
except Exception as e:
    logging.error(f"Failed to detect duplicates: {e}")
    raise


INFO:Duplicate rows found: 0


Duplicate rows (keep all duplicates): 0


,order_id,customer_id,product_id,order_date,quantity,unit_price,region,revenue


## 3.3 Algorithm 3 — Rolling Average Revenue (7-day)

Daily revenue is often noisy.  
A 7-day rolling average smooths the trend and makes abnormal spikes easier to notice.


In [34]:
try:
    roll7 = rolling_avg_revenue(df, window_days=7)
    display(roll7.tail(10))
except Exception as e:
    logging.error(f"Failed to compute rolling average revenue: {e}")
    raise


INFO:Computed rolling average revenue with window_days=7.


,date,revenue,rolling_avg_revenue
890,2024-06-09,64891.99,66450.101429
891,2024-06-10,65441.40,67300.744286
892,2024-06-11,54852.71,66888.445714
893,2024-06-12,63126.67,67713.711429
894,2024-06-13,68778.83,65876.268571
895,2024-06-14,66219.43,65219.911429
896,2024-06-15,74291.07,65371.728571
897,2024-06-16,72799.17,66501.325714
898,2024-06-17,59438.91,65643.827143
899,2024-06-18,64830.86,67069.277143


## 3.4 Algorithm 4 — Z-Score Anomaly Detection

We flag transactions with unusually high revenue using a z-score threshold of 3.0.  
These are not “confirmed fraud”, but they are strong **fraud signals** that should be reviewed.


In [35]:
try:
    anoms = zscore_anomalies(df, z_thresh=3.0)
    display(anoms.head(10))
except Exception as e:
    logging.error(f"Failed to compute z-score anomalies: {e}")
    raise


INFO:Anomalies found (z_thresh=3.00): 0


,order_id,customer_id,product_id,order_date,quantity,unit_price,region,revenue,revenue_numeric,z_score


## 4) Secure Coding Reflection (Logging + Error Handling)

Logging helps trace what the system did and supports debugging and audits.  
Error handling prevents crashes and provides clear messages when files or data are invalid.  
In fraud detection, these practices are important because the system must be reliable and explainable.


## 5) Mini-Checks (Assertions)

Assertions are small tests to confirm the outputs make sense:
- Top-K returns the expected columns and row count
- Rolling average produces values
- Anomalies (if any) exceed the threshold


In [36]:
# 1) top-k shape + columns
assert isinstance(top10, pd.DataFrame)
assert set(["product_id", "revenue"]).issubset(top10.columns)
assert 1 <= len(top10) <= 10

# 2) duplicates is a DataFrame (and if not empty, they should be duplicates in the original df)
assert isinstance(dups, pd.DataFrame)
if len(dups) > 0:
    # Every returned row should appear more than once in the original dataset
    # (This is a safe property even if src uses keep=False)
    assert df.duplicated(keep=False).sum() >= len(dups)

# 3) rolling average has expected columns and some non-null rolling values
assert isinstance(roll7, pd.DataFrame)
assert set(["date", "revenue", "rolling_avg_revenue"]).issubset(roll7.columns)
assert roll7["rolling_avg_revenue"].notna().any()

# 4) anomalies must exceed threshold if any exist
assert isinstance(anoms, pd.DataFrame)
if len(anoms) > 0:
    assert (anoms["z_score"] > 3.0).all()

logging.info("ALL ASSERTIONS PASSED")


INFO:ALL ASSERTIONS PASSED


We learned that on large datasets, we must use efficient methods like groupby, sorting, rolling averages, and vectorized math, because slow loops don't scale well. The most difficult step was loading and cleaning the data, and logging helped by showing clear messages about what loaded, the dataset size, and exactly where errors happened. This project prepares us for ML later because it builds the basics of ML workflows such as clean data and feature selection, made it safe, repeatable pipeline before training models.